Alex -Net Implementation

In [ ]:
import torch
import torch.nn.functional as F
import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
import torch.nn as nn
import torch.utils.data as data
from torch.utils.data import DataLoader

In [ ]:
import numpy as np
from tqdm import tqdm
import time
import os

In [ ]:
data_path = "./data_path"
checkpoint_path = "./checkpoint_path"

os.makedirs(checkpoint_path, exist_ok = True)

def train_seed(seed):
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

train_seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available()else torch.device("cpu")

print("using this device", device)

using this device cuda:0


In [ ]:
train_dataset = CIFAR10(root = data_path, download = True)
train_std = ((train_dataset.data/255.0).std(axis= (0,1,2)))
train_mean = ((train_dataset.data/255.0).mean(axis = (0,1,2)))

print(train_std)
print(train_mean)

[0.24703223 0.24348513 0.26158784]
[0.49139968 0.48215841 0.44653091]


In [ ]:
train_transform = transforms.Compose([transforms.ToTensor(),
                                      transforms.RandomHorizontalFlip(p = 0.5),
                                      transforms.RandomResizedCrop((32, 32), ratio = (0.2, 0.8), scale = (0.2, 0.8)),
                                      transforms.Normalize(train_std, train_mean)])

test_transform = transforms.Compose([transforms.ToTensor(),
                                                        transforms.Normalize(train_std, train_mean)])



train_dataset = CIFAR10(root = data_path, train = True, download = True, transform = train_transform)
test_dataset = CIFAR10(root = data_path,  train = False, download = True, transform = test_transform)
val_dataset = CIFAR10(root = data_path, download = True, train = True, transform = train_transform)



train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size


train_set, _ = torch.utils.data.random_split(train_dataset, [train_size, val_size])
_, val_set = torch.utils.data.random_split(val_dataset, [train_size, val_size])



train_loader = data.DataLoader(train_set, batch_size = 256, shuffle = True, pin_memory =True, num_workers = 1)
val_loader = data.DataLoader(val_set, batch_size = 256, shuffle = False, pin_memory = True,num_workers = 1)
test_loader = data.DataLoader(test_dataset, batch_size = 256, shuffle = False, pin_memory = True, num_workers = 1)


print(train_loader)
print(val_loader)
print(test_loader)




In [ ]:
class AlexNet(nn.Module):
  def __init__(self):
    super(AlexNet, self).__init__()


    self.input_net = nn.Sequential(
        nn.Conv2d(in_channels = 3, out_channels = 96, kernel_size= 3,  stride =1, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        nn.Conv2d(in_channels = 96, out_channels = 256, kernel_size = 3, stride = 1, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        nn.Conv2d(in_channels = 256, out_channels = 396, kernel_size = 3, stride = 1, padding = 1),
        nn.ReLU(),
        nn.Conv2d(in_channels = 396, out_channels = 396, kernel_size = 3, stride = 1, padding = 1),
        nn.ReLU(),
        nn.Conv2d(in_channels = 396, out_channels = 256,  kernel_size = 3, stride = 1, padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2)
    )


    self.fc1 = nn.Linear(in_features = 256 * 4 * 4, out_features =4092)
    self.fc2 = nn.Linear(in_features =4092, out_features = 4092)
    self.fc3 = nn.Linear(in_features = 4092, out_features = 10)

  def forward(self, x):
    out = self.input_net(x)
    out = out.reshape(out.shape[0], -1)
    out = self.fc1(out)
    out = self.fc2(out)
    out = self.fc3(out)
    return out

In [ ]:
model = AlexNet().to(device)
model

AlexNet(
  (input_net): Sequential(
    (0): Conv2d(3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(96, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(256, 396, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): Conv2d(396, 396, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): Conv2d(396, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU()
    (12): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Linear(in_features=4096, out_features=4092, bias=True)
  (fc2): Linear(in_features=4092, out_features=4092, bias=True)
  (fc3): Linear(in_features=4092, out_features=100, bias=True)
)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
num_epochs = 100
for epochs in tqdm(range(num_epochs)):
  for data_input, data_labels in train_loader:
    data_input = data_input.to(device)
    data_label = data_labels.to(device)

    optimizer.zero_grad()
    pred = model(data_input)
    loss = criterion(pred, data_label)


    loss.backward()
    optimizer.step()

print(f'Epoch: [{epochs + 1} / {num_epochs}], Loss: {loss.item():.4f}')

100%|██████████| 100/100 [43:53<00:00, 26.33s/it]

Epoch: [100 / 100], Loss: 0.9703


In [ ]:
with torch.no_grad():
  correct = 0
  total = 0
  for data_input, data_labels in test_loader:
    data_input = data_input.to(device)
    data_label = data_labels.to(device) # data_labels moved to device and stored in data_label

    pred = model(data_input)
    _, predicted = torch.max(pred.data, 1)
    total += data_labels.size(0)
    correct += (predicted == data_label).sum().item() # Use data_label for comparison


  acc = 100 * correct / total
  print(f"Accuracy of the values of the test_model is {acc:.3f}%")

Accuracy of the values of the test_model is 73.560%
